In [ ]:
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sys.path.append('../src')

from evaluate_results import extract_long_campaign_results

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
df_campaign_raw = pd.read_csv('../data/campaign_results.csv')
df_segmented = pd.read_csv('../results/profiles_segmented.csv')

df_campaign, comparison_available, warnings = extract_long_campaign_results(df_campaign_raw)
df_full = df_campaign.merge(df_segmented, on='id_profil', how='left')

print(f'Comparaison generique/personnalisee disponible : {comparison_available}')
if warnings:
    print('Avertissements :')
    for warning in warnings:
        print('-', warning)

display(df_full.head())

In [ ]:
metric_columns = [
    column for column in [
        'taux_ouverture',
        'taux_clic',
        'taux_reponse',
        'delai_reaction_moyen_heures',
    ] if column in df_campaign.columns
]

campaign_summary = df_campaign.groupby('condition')[metric_columns].mean().round(4)
display(campaign_summary)

In [ ]:
if {'generique', 'personnalisee'}.issubset(set(df_campaign['condition'].unique())):
    chart_df = campaign_summary.reset_index().melt(
        id_vars='condition',
        var_name='metrique',
        value_name='valeur',
    )

    ax = sns.barplot(data=chart_df, x='metrique', y='valeur', hue='condition', palette='viridis')
    ax.set_title('Comparaison generique vs personnalisee')
    ax.set_xlabel('Metrique')
    ax.set_ylabel('Valeur moyenne')
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()
else:
    print('Le fichier campaign_results.csv ne separe pas encore les conditions generique et personnalisee.')

In [ ]:
if 'segment_principal' in df_full.columns and not df_full['segment_principal'].isna().all():
    segment_summary = df_full.groupby('segment_principal')[['taux_clic']].mean().sort_values('taux_clic', ascending=False)
    display(segment_summary)

    sns.barplot(data=segment_summary.reset_index(), x='taux_clic', y='segment_principal', palette='magma')
    plt.title('Taux de clic moyen par segment')
    plt.xlabel('Taux de clic moyen')
    plt.ylabel('Segment principal')
    plt.tight_layout()
    plt.show()
else:
    print('Le fichier profiles_segmented.csv doit etre regenere sur le dataset final pour une analyse par segment fiable.')